In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src" / "config.py").exists() and root != root.parent:
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

import config
print("Project:", config.ROOT)

Project: C:\Users\jtrin\Documents\Ironhack\Quality air


In [2]:
import collect_flat_files
collect_flat_files.main()

1) Airparif station files
   2018 CO.csv              5 columns,    38142 measurements
   2018 NO2.csv            41 columns,   334783 measurements
   2018 NOX.csv            41 columns,   318140 measurements, no data at all: ['BAGN', 'PA01H']
   2018 O3.csv             24 columns,   206279 measurements
   2018 PM10.csv           24 columns,   191280 measurements
   2018 PM25.csv           13 columns,   109605 measurements
   2019 CO.csv              6 columns,    32550 measurements, no data at all: ['BASCH']
   2019 NO2.csv            41 columns,   328389 measurements, no data at all: ['BASCH']
   2019 NOX.csv            40 columns,   311806 measurements, no data at all: ['BAGN', 'BASCH']
   2019 O3.csv             25 columns,   204885 measurements
   2019 PM10.csv           25 columns,   182736 measurements, no data at all: ['BASCH']
   2019 PM25.csv           14 columns,   107995 measurements
   2020 CO.csv              5 columns,    41761 measurements
   2020 NO2.csv            40 

In [3]:
import scrape_airparif
scrape_airparif.main()

1) HTML page of the current year
  robots.txt read (HTTP 200)
   27 rows extracted
2) PDF reports of past years
  [!] https://www.airparif.fr/sites/default/files/Bilan_Episodes_Web_2018.pdf -> HTTP 404
   2018: no PDF at this address (to look for by hand)
  [!] https://www.airparif.fr/sites/default/files/Bilan_Episodes_Web_2019.pdf -> HTTP 404
   2019: no PDF at this address (to look for by hand)
  [!] https://www.airparif.fr/sites/default/files/Bilan_Episodes_Web_2020.pdf -> HTTP 404
   2020: no PDF at this address (to look for by hand)
  [!] https://www.airparif.fr/sites/default/files/Bilan_Episodes_Web_2021.pdf -> HTTP 404
   2021: no PDF at this address (to look for by hand)
  [!] https://www.airparif.fr/sites/default/files/Bilan_Episodes_Web_2022.pdf -> HTTP 404
   2022: no PDF at this address (to look for by hand)
  [!] https://www.airparif.fr/sites/default/files/Bilan_Episodes_Web_2023.pdf -> HTTP 404
   2023: no PDF at this address (to look for by hand)
   2024: 7 rows extracte

In [5]:
import api_meteofrance
api_meteofrance.main()

1) Yearly orders for station 75114001 (Paris-Montsouris)
   2018: order no. 2026035959977
      file not ready yet (attempt 1), waiting 10 s
      365 rows received
   2019: order no. 2026035959988
      file not ready yet (attempt 1), waiting 10 s
      365 rows received
   2020: order no. 2026035959999
      file not ready yet (attempt 1), waiting 10 s
      366 rows received
   2021: order no. 2026035960005
      file not ready yet (attempt 1), waiting 10 s
      365 rows received
   2022: order no. 2026035960087
      file not ready yet (attempt 1), waiting 10 s
      365 rows received
   2023: order no. 2026035960118
      file not ready yet (attempt 1), waiting 10 s
      365 rows received
   2024: order no. 2026035960137
      file not ready yet (attempt 1), waiting 10 s
      366 rows received
   2025: order no. 2026035960150
      file not ready yet (attempt 1), waiting 10 s
      365 rows received
   2026: order no. 2026035960158
      file not ready yet (attempt 1), waiting 

In [6]:
import build_dataset
build_dataset.main()

1) Pollutants measured in central Paris (PA01H then PA04C)
   PM2.5 days per station: {'PA01H': 2380, 'PA04C': 575}
   columns: ['co', 'no2', 'nox', 'o3', 'pm10', 'pm25', 'o3_max']
2) Complete calendar (every day, even without measurement)
3) Weather
4) Calendar variables
5) Official Airparif episodes (scraping + Open Data), one flag per pollutant
   episode days per pollutant: {'episode_pm10': 49, 'episode_o3': 58, 'episode_no2': 0}
6) Binary target: exceedance of the WHO threshold

3186 days (01/01/2018 -> 21/09/2026) -> C:\Users\jtrin\Documents\Ironhack\Quality air\data\processed\dataset_daily.csv
Missing PM2.5: 231 days (7.3%)
Days above the WHO threshold: 658 (22.3% of measured days)


In [7]:
import pandas as pd

dataset = pd.read_csv(config.DATA_PROCESSED / "dataset_daily.csv", parse_dates=["date"])
print(dataset.shape)
dataset.head()

(3186, 27)


,date,co,no2,nox,o3,pm10,pm25,o3_max,source_station,temp_mean_c,...,wind_dir_deg,day_of_week,month,is_weekend,is_public_holiday,is_school_holiday,episode_pm10,episode_o3,episode_no2,above_who
0,2018-01-01,NaN,18.39,20.70,61.96,11.26,6.04,70.0,PA04C,8.0,...,300.0,0,1,0,1,1,0,0,0,0.0
1,2018-01-02,NaN,32.67,40.60,39.29,15.59,8.29,67.0,PA04C,8.7,...,230.0,1,1,0,0,1,0,0,0,0.0
2,2018-01-03,NaN,20.21,23.08,60.71,15.73,9.17,77.0,PA04C,9.9,...,310.0,2,1,0,0,1,0,0,0,0.0
3,2018-01-04,NaN,29.79,36.01,40.67,9.79,6.58,59.0,PA04C,11.6,...,260.0,3,1,0,0,1,0,0,0,0.0
4,2018-01-05,NaN,41.46,49.18,42.87,14.44,NaN,76.0,NaN,8.7,...,220.0,4,1,0,0,1,0,0,0,NaN


In [9]:
import load_mysql
load_mysql.main()

1) Preparing the tables in pandas
   station                  52 rows
   pollutant                 6 rows
   daily_measurement    386561 rows
   calendar               3186 rows
   weather_day            3186 rows
   episode                 107 rows
2) Creating the database (if needed)
3) Creating the tables (sql/schema.sql)
   12 statements run from schema.sql
4) Inserting: dimensions first, facts next
   station                  52 rows inserted
   pollutant                 6 rows inserted
   calendar               3186 rows inserted
   weather_day            3186 rows inserted
   daily_measurement    386561 rows inserted
   episode                 107 rows inserted
5) Checking the row counts
   station            MySQL =       52   pandas =       52   OK
   pollutant          MySQL =        6   pandas =        6   OK
   calendar           MySQL =     3186   pandas =     3186   OK
   weather_day        MySQL =     3186   pandas =     3186   OK
   daily_measurement  MySQL =   386561  

In [10]:
import export_bigquery

In [11]:
export_bigquery.main()

1) Rebuilding the MySQL tables in pandas
2) Joining all the tables into one

3) 386561 rows x 27 columns -> C:\Users\jtrin\Documents\Ironhack\Quality air\data\processed\bigquery\daily_measurements_denorm.csv (58 MB)
   period: 2018-01-01 -> 2026-09-21
   schema to paste in the console (Edit as text): C:\Users\jtrin\Documents\Ironhack\Quality air\data\processed\bigquery\bigquery_schema.json
   OK: under the 100 MB limit of a browser upload.
